# 02_pure_wall_master: independent resumable A100 bundle

S1/T1/B2 master trajectories, live-state H1/H2 descendants, exact G5 fixed-word spectra, and supplemental R1 record statistics. This notebook launches only `02_pure_wall_master`; it never continues into another experiment. Every ordinary stochastic case uses 10 independent trajectories in two immutable five-trajectory shards. Run the default report-only pass first, inspect the exact verified/pending inventory, then set `RESUME_REPORT_ONLY=False` to compute only missing shards.

## 1. Mount Drive and locate the campaign package


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

MYDRIVE = Path('/content/drive/MyDrive')
CAMPAIGN_ROOT = MYDRIVE / 'final_production_ready_figure_scripts'
runner = CAMPAIGN_ROOT / 'colab_bundle_runner.py'
bundle_root = CAMPAIGN_ROOT / 'prior_designs' / '02_pure_wall_master'
if not runner.is_file() or not (bundle_root / 'run_bundle.py').is_file():
    raise FileNotFoundError('Upload the complete updated final_production_ready_figure_scripts folder')
bundle_config = json.loads((bundle_root / 'production_config.json').read_text())
OUTPUT_BUNDLE = bundle_config.get('output_bundle', bundle_root.name)
print(f'[campaign] {CAMPAIGN_ROOT}')
print(f'[bundle] {bundle_root}')


## 2. Select this bundle's queue


In [ ]:
RUN_PROFILE = 'production'
BUNDLE = '02_pure_wall_master'
CASE_PREFIXES = ()  # e.g. ('S2_',); empty runs every currently eligible case
CASE_IDS = ()       # exact IDs; empty uses prefixes/full bundle
MAX_SESSION_GPU_HOURS_OVERRIDE = None
PREFLIGHT_ONLY = False
RESUME_REPORT_ONLY = True  # inspect first; change to False to launch missing work
HEARTBEAT_SECONDS = 60
RUN_QUEUE = True
if HEARTBEAT_SECONDS <= 0:
    raise ValueError('HEARTBEAT_SECONDS must be positive')
if CASE_PREFIXES and CASE_IDS:
    raise ValueError('use CASE_PREFIXES or CASE_IDS, not both')
print(json.dumps({
    'bundle': BUNDLE, 'profile': RUN_PROFILE, 'case_prefixes': CASE_PREFIXES,
    'case_ids': CASE_IDS, 'resume_report_only': RESUME_REPORT_ONLY,
    'heartbeat_seconds': HEARTBEAT_SECONDS,
}, indent=2))


## 3. Verify, resume, and stop at the end of this bundle


In [ ]:
if not RUN_QUEUE:
    raise RuntimeError('RUN_QUEUE is False')
command = [
    sys.executable, '-u', str(runner), '--drive-root', str(MYDRIVE),
    '--bundle', BUNDLE, '--profile', RUN_PROFILE,
    '--heartbeat-seconds', str(HEARTBEAT_SECONDS),
]
for prefix in CASE_PREFIXES:
    command += ['--case-prefix', str(prefix)]
for case_id in CASE_IDS:
    command += ['--case-id', str(case_id)]
if MAX_SESSION_GPU_HOURS_OVERRIDE is not None:
    command += ['--max-session-hours', str(MAX_SESSION_GPU_HOURS_OVERRIDE)]
if PREFLIGHT_ONLY:
    command.append('--preflight-only')
if RESUME_REPORT_ONLY:
    command.append('--resume-report-only')
production_collection = 'classA_final_production_outputs/production_10sample_v4_occupied_frame_cycle_resolved'
pilot_collection = 'classA_pilot_outputs/production_10sample_v4_occupied_frame_cycle_resolved'
collection = production_collection if RUN_PROFILE == 'production' else pilot_collection
output_root = MYDRIVE / collection
session_dir = output_root / '_bundle_sessions'
build_id = hashlib.sha256(runner.read_bytes()).hexdigest()[:16]
print('[bundle dashboard]')
print(json.dumps({
    'bundle': BUNDLE, 'profile': RUN_PROFILE, 'output_root': str(output_root),
    'runner_build_id': build_id, 'heartbeat_seconds': HEARTBEAT_SECONDS,
    'session_dir': str(session_dir), 'case_prefixes': CASE_PREFIXES,
    'case_ids': CASE_IDS, 'resume_report_only': RESUME_REPORT_ONLY,
}, indent=2))
print(f'[launch bundle] {" ".join(command)}', flush=True)
result = subprocess.run(command, check=False)
if result.returncode:
    sessions = sorted(session_dir.glob(f'*_{BUNDLE}_*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    if sessions:
        latest = json.loads(sessions[0].read_text())
        print('[saved bundle failure]')
        print(json.dumps({'session': str(sessions[0]), 'failure': latest.get('failure'), 'current': latest.get('current'), 'log': latest.get('session_log')}, indent=2))
    raise subprocess.CalledProcessError(result.returncode, command)


## 4. Optional supplemental record large-deviation summary


In [ ]:
RUN_R1_ANALYSIS = False
if RUN_R1_ANALYSIS:
    archive_root = output_root / OUTPUT_BUNDLE
    analysis_root = archive_root / 'R1_analysis'
    subprocess.run([sys.executable, '-u', str(bundle_root / 'src' / 'record_spectrum_analysis.py'), '--archive-root', str(archive_root), '--output-root', str(analysis_root)], check=True)
    print((analysis_root / 'R1_record_spectrum_summary.json').read_text())


## 5. Exact G5 fixed-word many-body singular spectrum

This deterministic descendant reuses completed primary explicit-interface and matched-trivial master words. It initializes the one-particle correlation matrix at $\mathbf{1}/2$ only as an algebraic probe of $\widehat K\widehat K^\dagger/Z_K$. It never propagates a Choi state, never changes a parent archive, and resumes from checksum-verified G5 receipts.

### Formula dictionary

For $N_{\rm orb}=2N_xN_y$, the normalized positive operator and its identity-input normalization are
$$\rho_R^K=\frac{\widehat K\widehat K^\dagger}{Z_K},\qquad p_K(\mathbf{1}/2)=2^{-N_{\rm orb}}Z_K,\qquad \log Z_K=N_{\rm orb}\log2+\log p_K.$$
If $\nu_i$ are the natural occupations of $\rho_R^K$, then
$$j_i=2\sqrt{\nu_i(1-\nu_i)},\qquad \mu_i^\pm=\frac{1\pm\sqrt{1-j_i^2}}2,$$
and the complete normalized many-body eigenvalues and absolute singular values are
$$\lambda_{\boldsymbol n}=\prod_i\nu_i^{n_i}(1-\nu_i)^{1-n_i},\qquad \sigma_{\boldsymbol n}=\sqrt{Z_K\lambda_{\boldsymbol n}}.$$
Relative to the dominant occupation $n_i^{(0)}=\mathbf{1}_{\nu_i\geq1/2}$, a flip carries $q_i=1-2n_i^{(0)}$ and singular-amplitude gap
$$\delta_i=\frac12\left|\log\frac{\nu_i}{1-\nu_i}\right|=\operatorname{arcosh}(1/j_i).$$
The final finite-size tests use
$$\Delta_{q,n}(L)=\frac{2\pi v}{L}\left(\frac{q^2}{2k}+n\right)+O(L^{-2}),\qquad f_K(L)=f_\infty-\frac{\pi c_{\rm eff}v}{12L^2}+O(L^{-4}),$$
where $f_K$ is formed after matched-trivial subtraction and division by the two walls. Population gaps $|\log[\nu_i/(1-\nu_i)]|$ are twice the coherent singular-amplitude gaps $\delta_i$.


In [ ]:
RUN_G5_MANYBODY_ANALYSIS = False
if RUN_G5_MANYBODY_ANALYSIS:
    archive_roots = [
        output_root / OUTPUT_BUNDLE,
        MYDRIVE / 'classA_final_production_outputs' / 'production_10sample_v3_fixed_nx20_ny20_30_40_50_60' / OUTPUT_BUNDLE,
    ]
    analysis_root = output_root / OUTPUT_BUNDLE / 'G5_manybody_spectrum'
    command = [sys.executable, '-u', str(bundle_root / 'src' / 'g5_manybody_spectrum_analysis.py'), '--bundle-root', str(bundle_root), '--output-root', str(analysis_root), '--bootstrap-replicates', '2000', '--mode', RUN_PROFILE]
    for archive_root in archive_roots:
        command += ['--archive-root', str(archive_root)]
    subprocess.run(command, check=True)
    print((analysis_root / 'G5_manybody_spectrum_summary.json').read_text())


## Final. Disconnect this completed Colab runtime


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
